# 2G vs 1G — Decoding protein embeddings

**M2 Bioinformatics — Assa Diabira — 2026**

This notebook visualizes and compares the performance of 2G models (ESM-C, Ankh2) and 1G baselines (ESM2, Ankh-Large, ProtT5) on the per-residue and global prediction tasks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path('../..')
RESULTS = ROOT / 'results'
FIGS = RESULTS / 'figures'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150

## 1. Load results

In [ ]:
df_2g = pd.read_csv(RESULTS / 'dt_results_2g.csv')
df_test = df_2g[df_2g['split'] == 'test'].copy()

# add generation column
gen_map = {'esmc_300m': '2G', 'esmc_600m': '2G', 'ankh2_large': '2G'}
df_test['generation'] = df_test['model'].map(gen_map)

# display labels
model_labels = {
    'esmc_300m': 'ESM-C 300M',
    'esmc_600m': 'ESM-C 600M',
    'ankh2_large': 'Ankh2-Large',
}
df_test['model_label'] = df_test['model'].map(model_labels)

task_order = ['rmsf', 'neq', 'bfact', 'acc', 'sec3', 'sec8']
task_labels = {
    'rmsf': 'RMSF\n(flexibility)',
    'neq': 'Neq\n(disorder)',
    'bfact': 'Bfactor',
    'acc': 'Solvent\naccessibility',
    'sec3': 'SS3',
    'sec8': 'SS8',
}
df_test['task_label'] = df_test['task'].map(task_labels)

print(df_test[['model_label', 'task', 'F1', 'MCC']].pivot_table(
    index='task', columns='model_label', values='F1'
).round(3))

## 2. F1 comparison — all 2G models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['F1', 'MCC']):
    pivot = df_test.pivot_table(index='task', columns='model_label', values=metric)
    pivot = pivot.reindex(task_order)
    pivot.plot(kind='bar', ax=ax, width=0.7)
    ax.set_title(f'{metric} — test set (Decision Tree)', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel(metric)
    ax.set_xticklabels([task_labels[t] for t in task_order], rotation=30, ha='right')
    ax.set_ylim(0, 1)
    ax.legend(title='Model', fontsize=9)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
plt.savefig(FIGS / 'comparison_2g_F1_MCC.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. F1 heatmap — 2G

In [ ]:
pivot_f1 = df_test.pivot_table(index='task', columns='model_label', values='F1')
pivot_f1 = pivot_f1.reindex(task_order)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot_f1, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0.15, vmax=0.9, ax=ax, linewidths=0.5)
ax.set_title('F1 score - test set - 2G models', fontsize=13)
ax.set_yticklabels(task_order, rotation=0)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIGS / 'heatmap_f1_2g.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Figures PCA

In [ ]:
from IPython.display import Image, display
import os

pca_figs = sorted(FIGS.glob('*_pca_*.png'))
print(f'{len(pca_figs)} PCA figures found')

for f in pca_figs[:6]:
    print(f.name)
    display(Image(str(f), width=500))

## 5. 2G vs 1G comparison (per-residue baselines)

In [ ]:
# load 1G results when available
results_1g_path = RESULTS / 'dt_results_1g.csv'

if results_1g_path.exists():
    df_1g = pd.read_csv(results_1g_path)
    df_all = pd.concat([df_2g, df_1g])
    df_all_test = df_all[df_all['split'] == 'test']
    
    pivot_all = df_all_test.pivot_table(index='task', columns='model', values='F1')
    print(pivot_all.round(3))
else:
    print('1G results not available yet — lancer scripts/run_embeddings_1g.sh')
    print('then build_1g_datasets.sh and run_analyses_1g.sh')

# Part 2 — GLOBAL properties (per-protein, WP2)

Core of the project: predicting **per-protein** properties from embeddings aggregated by **mean pooling**.

7 tasks: `fold_label` (SCOP a/b/c/d, macro-F1), `localization_class` (5 classes, macro-F1), `species_label` (4 organisms, macro-F1), `tm_label` (binary, MCC), `disorder_global` (binarized mean RMSF, MCC), `acc_mean` (binarized mean accessibility, MCC), `aggregation_score` (A3D-like structural proxy, MCC).

Classifiers: LogReg / RandomForest / MLP — 15-fold CV on train, final evaluation on the test set.

In [ ]:
df_g = pd.read_csv(RESULTS / 'global_results.csv')

MODEL_ORDER = ['esmc_300M','esmc_600M','ankh2_large',
               'esm2_t33_650M_UR50D','ankh_large','prot_t5_xl_uniref50']
MODEL_LABELS = {'esmc_300M':'ESM-C 300M','esmc_600M':'ESM-C 600M','ankh2_large':'Ankh2-Large',
                'esm2_t33_650M_UR50D':'ESM2-650M','ankh_large':'Ankh-Large 1G','prot_t5_xl_uniref50':'ProtT5'}
GEN = {'esmc_300M':'2G','esmc_600M':'2G','ankh2_large':'2G',
       'esm2_t33_650M_UR50D':'1G','ankh_large':'1G','prot_t5_xl_uniref50':'1G'}
TASK_ORDER = ['fold_label','localization_class','tm_label','disorder_global','acc_mean']

# best classifier per (model, task) by score_test
best_g = df_g.loc[df_g.groupby(['model','task'])['score_test'].idxmax()].copy()
best_g['model_label'] = best_g['model'].map(MODEL_LABELS)
best_g['gen'] = best_g['model'].map(GEN)
print(best_g[['model_label','task','clf','metric','score_cv_mean','score_test']]
      .sort_values(['task','model_label']).to_string(index=False))

## Heatmap — best test score per model x task

In [ ]:
piv = best_g.pivot_table(index='task', columns='model', values='score_test')
piv = piv.reindex(index=TASK_ORDER, columns=MODEL_ORDER)
piv.columns = [MODEL_LABELS[c] for c in piv.columns]

fig, ax = plt.subplots(figsize=(9,5))
sns.heatmap(piv, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0.2, vmax=0.8,
            linewidths=0.5, ax=ax)
ax.set_title('Global properties - best test score (macro-F1 / MCC)')
ax.set_xlabel(''); ax.set_ylabel('')
plt.axvline(3, color='black', lw=2)  # 2G | 1G separator
plt.tight_layout()
plt.savefig(FIGS / 'global_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 2G vs 1G comparison — per task (best of each generation)

In [ ]:
rows = []
for task in TASK_ORDER:
    for gen in ['2G','1G']:
        sub = best_g[(best_g['task']==task) & (best_g['gen']==gen)]
        if len(sub):
            top = sub.loc[sub['score_test'].idxmax()]
            rows.append({'task':task,'gen':gen,'score':top['score_test'],
                         'model':MODEL_LABELS[top['model']]})
cmp = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9,5))
piv2 = cmp.pivot(index='task', columns='gen', values='score').reindex(TASK_ORDER)
piv2.plot(kind='bar', ax=ax, color={'2G':'#4C72B0','1G':'#C44E52'}, width=0.7)
ax.set_title('Best 2G vs 1G model per global task')
ax.set_ylabel('score test'); ax.set_xlabel(''); ax.set_ylim(0,1)
ax.set_xticklabels(TASK_ORDER, rotation=25, ha='right')
ax.legend(title='Generation')
plt.tight_layout()
plt.savefig(FIGS / 'global_2g_vs_1g.png', dpi=200, bbox_inches='tight')
plt.show()
print(piv2.round(3))

## SHAP & drop-curve figures

Generated by `analysis/global/shap_drop_global.py` into `results/figures/global_shap_*` and `global_drop_*`.

In [ ]:
from IPython.display import Image, display
shap_figs = sorted(FIGS.glob('global_shap_*.png'))
drop_figs = sorted(FIGS.glob('global_drop_*.png'))
print(f'{len(shap_figs)} figures SHAP, {len(drop_figs)} drop curves')
for f in shap_figs[:3] + drop_figs[:3]:
    print(f.name); display(Image(str(f), width=460))